# Test chunk building and token lengths

In [ ]:
import json
from kb_setup.chunker import build_chunks, _count_tokens
from kb_setup.indexer import log_chunk_stats

CONTENT_LIST_PATH = "../data/docs/outputs/<your_doc>/auto/<your_doc>_content_list.json"

with open(CONTENT_LIST_PATH, encoding="utf-8") as f:
    content = json.load(f)
print(f"Loaded {len(content)} raw elements")

chunks = build_chunks(content)
log_chunk_stats(chunks)

print()
for c in chunks:
    tokens = _count_tokens(c["embed_text"])
    status = "✓" if tokens <= 480 else "✗ OVER"
    print(f"{status}  {tokens:>4} tokens  {c['heading'][:60]}")

# Query and retrieve chunks

In [ ]:
from doc_registry import collection_name_for, list_all
from retriever import query_documents, format_chunks_as_context
import json

# List all indexed documents
print("Indexed documents:")
for doc in list_all():
    print(f"  {doc['filename']}  ->  {doc['collection_name']}  ({doc['chunk_count']} chunks)")

In [ ]:
FILENAME = "/Users/mayurgd/Documents/CodingSpace/adobe_task_1/data/docs/inputs/d164e2f9-27b8-4963-a1ae-85e506a4c762_q12025.pdf"  # change this
COLLECTION = collection_name_for(FILENAME)

query = "what are the targets for second quarter"
results = query_documents(query, collection_name=COLLECTION, top_k=3)

for r in results:
    print(f"[{r['rank']}] score={r['score']}  pages={r['pages']}")
    print(f"  heading : {r['heading']}")
    print(f"  text    : {r['text']}")
    print()

In [ ]:
# Inspect a specific chunk in full
import json
for r in results:
    if r['heading'] == "Financial Targets":
        print(json.dumps(r, indent=2))

In [ ]:
# View formatted context as the LLM will see it
print(format_chunks_as_context(results))

# TEST RETRIEVAL ANSWERS

In [ ]:
from answer_query import answer_query, print_result
from kb_setup.doc_registry import collection_name_for

result = answer_query(
    query="What are the targets for Q2?",
    collection_name=collection_name_for("d164e2f9-27b8-4963-a1ae-85e506a4c762_q12025.pdf"),
    top_k=5,
)
print_result(result)
# or access result.answer and result.sources directly for your UI